# 06. 전체 실행 (한 사람이 한 GPU로 A·B·C·D + 베이스라인 + 분석)

모든 묶음을 한 세션에서 차례로 돌린다. 이미 끝난 실험은 건너뛰므로, 설정에 묶음이나 seed가 추가되면 이 노트북을 다시 모두 실행하면 새로 추가된 것만 돈다. **다른 사람이 같은 묶음을 동시에 돌리지 않게** 먼저 공지한다.

D 묶음(인기도 입력)과 B·C의 seed 43·44는 결과 확인 후 추가한 분석이다 (사전 등록 아님).

In [ ]:
# @title 0. 공통 준비 (모든 노트북 동일)
import sys
if any(m.startswith("msrs_paper") for m in sys.modules):
    raise RuntimeError("이전에 불러온 코드가 메모리에 남아 있습니다. 메뉴 '런타임 → 세션 다시 시작' 후 이 셀부터 다시 실행하세요.")
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/yuhwani/multimodal-recsys.git"  # upstream에 머지되면 3sunghynix 주소로 변경
BRANCH = "main"
!rm -rf /content/repo && git clone -q -b {BRANCH} {REPO_URL} /content/repo
!pip -q install -r /content/repo/paper/requirements.txt
!cd /content/repo && git log -1 --format="코드 커밋: %h %s (%cd)"

import sys
sys.path.insert(0, "/content/repo/paper")
from msrs_paper.config import load_all, Paths
cfg, exps, plan = load_all()
DRIVE_ROOT = cfg["drive_root"]  # 공유 폴더 바로가기 경로가 다르면 이 줄만 바꾼다
paths = Paths(DRIVE_ROOT).make_dirs()
print("드라이브 폴더:", paths.root)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# @title 1. 공유 준비물 확인 (체크섬이 다르면 멈춤)
from msrs_paper.utils import verify_manifest
m = verify_manifest(paths)
print(f"준비물 {len(m['files'])}개 일치 — 준비물 커밋 {m['git_commit']}")

In [ ]:
# @title 2. 전체 진행 현황
assert cfg["train"]["tuned"] and plan["status"] == "final"
GROUPS = list(exps["groups"])
def status():
    for g in GROUPS:
        runs = [(b["split"], c, s) for b in exps["groups"][g] for s in b["seeds"] for c in b["conds"]]
        done = [r for r in runs if (paths.run_dir(g, r[0], f"{r[1]}_seed{r[2]}") / "run_info.json").exists()]
        print(f"묶음 {g}: 전체 {len(runs)}회, 완료 {len(done)}회")
status()

In [ ]:
# @title 3. 전체 학습 (끊기면 0번 셀부터 다시 모두 실행하면 이어서)
import time
from msrs_paper import two_tower
from msrs_paper.utils import now
log_file = paths.logs / "run_all.log"
def log(msg):
    print(msg)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"{now()} {msg}\n")
t0 = time.time()
for g in GROUPS:
    log(f"===== 묶음 {g} =====")
    two_tower.run_group(cfg, exps, paths, g, log=log)
    log(f"===== 묶음 {g} 완료 ({(time.time() - t0) / 60:.1f}분 경과) =====")
status()

In [ ]:
# @title 4. 베이스라인 B1 · B2, 학습 없는 방식 K0 · K2
from msrs_paper import baselines
for name in ["B1", "B2", "K0", "K2"]:
    baselines.run(cfg, paths, name, "main", group="C", log=log)

In [ ]:
# @title 5. 전체 채점 · 분석 (100회마다 진행 표시)
from msrs_paper import analysis
t1, contrib, overall, mt = analysis.run_all(cfg, exps, plan, paths, log=log)
print("\n주 검정:", mt)
t1

In [ ]:
# @title 6. 전체 이미지 기여도 (인코더 × 구간)
overall

In [ ]:
# @title 7. 주 검정 조건의 층화 기여도 (패턴 − 무지)
m = plan["main_test"]
contrib[(contrib["measure"] == m["measure"]) & (contrib["segment"] == m["segment"])]

In [ ]:
# @title 8. 그림 1 · 그림 2
from IPython.display import Image, display
display(Image(str(paths.results / "fig1.png")))
display(Image(str(paths.results / "fig2.png")))